# Healthcare KG · Population Intelligence V3
**Validation and observable-treatment-characteristic analysis** · NHANES 2017–2018 · isolated Colab workflow

**Purpose:** reconcile V2 denominators, independently check design-based uncertainty, and calculate *descriptive* overlaps of kidney function, albuminuria and serum potassium among adults meeting the V1 joint CKD-screen + diabetes proxy. **No Kerendia eligibility, prescription recommendation, incidence, or market-size claims.**

**Input:** your `drive-download-20261025T010034Z-1-001.zip` containing `baseline_kg_COPY.json` and V2 outputs. The notebook downloads the same four public NHANES 2017–2018 XPT files, caches them in a new workspace, and does **not** connect to GitHub or alter baseline KG. All statistics must be reviewed before use in market intelligence.

In [ ]:
%pip -q install pandas numpy scipy requests networkx pyreadstat

## 1. Upload V2 results ZIP and verify baseline integrity

In [ ]:
from pathlib import Path
import json, hashlib, zipfile, shutil, datetime
import pandas as pd, numpy as np, requests
from google.colab import files
ROOT=Path('/content/healthcare_kg_population_v3')
ROOT.mkdir(parents=True,exist_ok=True)
uploaded=files.upload()
zip_paths=[Path('/content')/name for name in uploaded if name.lower().endswith('.zip')]
assert zip_paths, 'Upload the V2 output ZIP containing baseline_kg_COPY.json and v2_validation_report.json'
archive=zip_paths[0]
with zipfile.ZipFile(archive) as z:
    required=['baseline_kg_COPY.json','v2_validation_report.json','v2_population_estimates.csv','v2_kidney_segmentation.csv','v2_source_manifest.csv']
    for name in required:
        matches=[x for x in z.namelist() if x.split('/')[-1]==name]
        assert len(matches)==1, f'Missing or duplicate {name}'
        (ROOT/name).write_bytes(z.read(matches[0]))
report_v2=json.loads((ROOT/'v2_validation_report.json').read_text())
kg_path=ROOT/'baseline_kg_COPY.json'
kg_hash=hashlib.sha256(kg_path.read_bytes()).hexdigest()
assert kg_hash==report_v2['baseline_sha256'], 'Baseline hash mismatch; stop.'
v2=pd.read_csv(ROOT/'v2_population_estimates.csv')
v2seg=pd.read_csv(ROOT/'v2_kidney_segmentation.csv')
print('Baseline SHA256:',kg_hash,' | V2 rows:',len(v2),' | segmentation rows:',len(v2seg))

## 2. Load exactly the original four NHANES 2017–2018 files
No new NHANES cycle is mixed into this validation. CDC download URLs and SHA-256 values are recorded.

In [ ]:
BASE='https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/'
SOURCES={k:BASE+k+'.xpt' for k in ['DEMO_J','ALB_CR_J','BIOPRO_J','DIQ_J']}
previous_manifest=pd.read_csv(ROOT/'v2_source_manifest.csv').set_index('dataset')
def get_xpt(name,url):
    path=ROOT/(name+'.xpt')
    if not path.exists():
        response=requests.get(url,timeout=120,headers={'User-Agent':'Academic-NHANES-Validation/3.0'})
        response.raise_for_status()
        assert len(response.content)>1000, f'Suspiciously small file {name}'
        path.write_bytes(response.content)
    sha=hashlib.sha256(path.read_bytes()).hexdigest()
    expected=str(previous_manifest.loc[name,'sha256'])
    assert sha==expected, f'{name} differs from V2 input; stop to investigate CDC file change.'
    return pd.read_sas(path,format='xport'),dict(dataset=name,source_url=url,sha256=sha,bytes=path.stat().st_size,cycle='2017-2018')
tables={}; manifest=[]
for name,url in SOURCES.items():
    tables[name],entry=get_xpt(name,url); manifest.append(entry)
pd.DataFrame(manifest).to_csv(ROOT/'v3_source_manifest.csv',index=False)
print({k:len(v) for k,v in tables.items()})

## 3. Reconstruct the V1/V2 cohort identically
The diabetes definition (self-report with diagnosis age ≥30) is a **proxy**, not a validated T2D classification. CKD screen uses a single eGFR and UACR measurement and does not establish ≥3-month chronicity. 2021 CKD-EPI creatinine equation is used. The NHANES 2017–2018 2-year MEC weights and masked design variables are retained.

In [ ]:
d=tables['DEMO_J'][['SEQN','RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVSTRA','SDMVPSU']].copy()
for name in ['ALB_CR_J','BIOPRO_J','DIQ_J']:
    d=d.merge(tables[name],on='SEQN',how='left',validate='one_to_one')
d=d.loc[d.RIDAGEYR>=20].copy()
cols=['RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVSTRA','SDMVPSU','URXUMA','URXUCR','LBXSCR','DIQ010','DIQ040']
assert all(c in d for c in cols)
for c in cols:d[c]=pd.to_numeric(d[c],errors='coerce')
female=d.RIAGENDR.eq(2); kap=np.where(female,.7,.9); alpha=np.where(female,-.241,-.302)
r=d.LBXSCR/kap
d['egfr']=142*np.minimum(r,1)**alpha*np.maximum(r,1)**(-1.2)*(.9938**d.RIDAGEYR)*np.where(female,1.012,1)
d.loc[d.LBXSCR.isna()|d.LBXSCR.le(0),'egfr']=np.nan
d['uacr']=100*d.URXUMA/d.URXUCR
d.loc[d.URXUCR.isna()|d.URXUCR.le(0)|d.URXUMA.isna(),'uacr']=np.nan
d['ckd_screen']=np.where(d.egfr.notna()&d.uacr.notna(),(d.egfr<60)|(d.uacr>=30),np.nan)
d['diabetes_reported']=np.where(d.DIQ010.isin([1,2]),d.DIQ010.eq(1),np.nan)
d['t2d_proxy']=np.where(d.diabetes_reported==0,0,np.where((d.diabetes_reported==1)&d.DIQ040.between(1,100),(d.DIQ040>=30).astype(float),np.nan))
d['joint']=np.where(pd.notna(d.ckd_screen)&pd.notna(d.t2d_proxy),(d.ckd_screen==1)&(d.t2d_proxy==1),np.nan)
d['potassium']=pd.to_numeric(d.LBXSKSI,errors='coerce') if 'LBXSKSI' in d else np.nan
d['age_group']=pd.cut(d.RIDAGEYR,[20,45,65,75,np.inf],right=False,labels=['20-44','45-64','65-74','75+'])
d['sex']=d.RIAGENDR.map({1:'Male',2:'Female'})
d['g_cat']=pd.cut(d.egfr,[-np.inf,15,30,45,60,90,np.inf],right=False,labels=['G5','G4','G3b','G3a','G2','G1'])
d['a_cat']=pd.cut(d.uacr,[0,30,300,np.inf],right=False,labels=['A1','A2','A3'])
design=d.WTMEC2YR.gt(0)&d.SDMVSTRA.notna()&d.SDMVPSU.notna()
complete=d.joint.notna()&design
print('Adults:',len(d),'complete:',int(complete.sum()),'positive:',int((d.joint==1).sum()))
assert int(complete.sum())==report_v2['joint_proxy_evaluable']
assert int((d.joint==1).sum())==report_v2['joint_proxy_cases']

## 4. Reconcile denominators (the V2 discrepancy)
**Two different denominators were used in V2:** complete-case adults (~216.8M weighted) for overall joint prevalence, versus all MEC adults (~238.7M weighted) for mutually exclusive segment prevalence. These are *different estimands*, not necessarily an arithmetic error. V3 reports both explicitly and uses the complete-case denominator consistently for its primary prevalence measures. Weighted counts of positive cases can match even when prevalence denominators differ.

In [ ]:
w=d.WTMEC2YR.fillna(0)
checks={
 'all_adults_MEC_weighted':float(w[design].sum()),
 'complete_case_adults_weighted':float(w[complete].sum()),
 'excluded_missing_adults_weighted':float(w[design&~complete].sum()),
 'joint_positive_weighted':float(w[complete & d.joint.eq(1)].sum()),
 'complete_case_unweighted':int(complete.sum()),
 'missing_unweighted':int((design&~complete).sum()),
 'positive_unweighted':int((complete&d.joint.eq(1)).sum()),
}
checks['prevalence_complete_case']=checks['joint_positive_weighted']/checks['complete_case_adults_weighted']
checks['prevalence_all_MEC_lower_bound_only']=checks['joint_positive_weighted']/checks['all_adults_MEC_weighted']
print(json.dumps(checks,indent=2))
assert abs(checks['joint_positive_weighted']-report_v2['population_estimate'])<2
pd.DataFrame([{'metric':k,'value':v} for k,v in checks.items()]).to_csv(ROOT/'v3_denominator_reconciliation.csv',index=False)

## 5. Design-based Taylor linearization with independent jackknife diagnostic
Variance uses the full adult survey design and retains out-of-domain observations as zero contributions. The second check uses leave-one-PSU-out, within-stratum pseudo-replicates for the **same weighted total**. Agreement is a useful implementation diagnostic, not an external validation of the NHANES design. Report approximate t intervals; do not use low-count segments for decisions.

In [ ]:
from scipy.stats import t as tdist
z=d.loc[design].copy(); z['weight']=z.WTMEC2YR
strata=sorted(z.SDMVSTRA.dropna().unique())
def taylor_var(contribution):
    tmp=z[['SDMVSTRA','SDMVPSU']].copy();tmp['v']=z.weight*contribution.reindex(z.index).fillna(0).astype(float)
    sums=tmp.groupby(['SDMVSTRA','SDMVPSU'],observed=True).v.sum()
    var=0.;dfree=0
    for _,part in sums.groupby(level=0):
        a=part.to_numpy();m=len(a)
        if m>1:var+=m/(m-1)*np.square(a-a.mean()).sum();dfree+=m-1
    return var,dfree
def jackknife_var_total(contribution):
    tmp=z[['SDMVSTRA','SDMVPSU']].copy();tmp['v']=z.weight*contribution.reindex(z.index).fillna(0).astype(float)
    sums=tmp.groupby(['SDMVSTRA','SDMVPSU'],observed=True).v.sum()
    var=0.
    for _,part in sums.groupby(level=0):
        a=part.to_numpy();m=len(a)
        if m>1:
            # Remove one PSU and reweight remaining PSUs by m/(m-1).
            pseudo=(a.sum()-a)*(m/(m-1))
            var+=(m-1)/m*np.square(pseudo-pseudo.mean()).sum()
    return var
positive=(complete&d.joint.eq(1)).astype(float)
primary_denom=complete.astype(float)
num=float((z.weight*positive.reindex(z.index)).sum()); den=float((z.weight*primary_denom.reindex(z.index)).sum())
var_count,dfree=taylor_var(positive)
var_jk=jackknife_var_total(positive)
p=num/den
resid=positive-p*primary_denom
var_resid,_=taylor_var(resid)
crit=tdist.ppf(.975,dfree)
main={'weighted_cases':num,'weighted_complete_case_denominator':den,'prevalence':p,
      'count_ci_low':max(0,num-crit*np.sqrt(var_count)),'count_ci_high':num+crit*np.sqrt(var_count),
      'prevalence_ci_low':max(0,p-crit*np.sqrt(var_resid)/den),
      'prevalence_ci_high':min(1,p+crit*np.sqrt(var_resid)/den),
      'design_df':dfree,'taylor_variance':var_count,'jackknife_diagnostic_variance':var_jk,
      'variance_relative_difference':abs(var_count-var_jk)/var_count if var_count else np.nan}
print(json.dumps(main,indent=2))
assert np.isclose(var_count,var_jk,rtol=1e-8), 'Independent variance implementations disagree'
old=v2[(v2.definition=='joint_t2d_proxy_v1')&(v2.segment=='All adults')].iloc[0]
assert abs(num-float(old.weighted_cases))<2
assert abs(p-float(old.prevalence))<1e-8

## 6. Missingness sensitivity bounds
Observed-case estimate is a **complete-case population count**, not a full-population diagnosis estimate. These two deliberately extreme scenarios assume every missing-case adult is negative or positive, respectively. They are **not confidence intervals**, and the upper bound is not a credible clinical forecast.

In [ ]:
missing=design&~complete
missing_w=float(w[missing].sum())
missingness=pd.DataFrame([
 {'scenario':'Missing all negative (lower extreme)','weighted_joint_cases':num,'adult_denominator':float(w[design].sum())},
 {'scenario':'Complete-case (primary analysis)','weighted_joint_cases':num,'adult_denominator':den},
 {'scenario':'Missing all positive (upper extreme)','weighted_joint_cases':num+missing_w,'adult_denominator':float(w[design].sum())},
])
missingness['prevalence']=missingness.weighted_joint_cases/missingness.adult_denominator
missingness.to_csv(ROOT/'v3_missingness_sensitivity.csv',index=False)
display(missingness)

## 7. Mutually exclusive CKD categories and *observable* treatment-related characteristics
The filters below are **descriptive research thresholds only**: eGFR ≥25 mL/min/1.73 m² and measured serum potassium ≤5.0 mmol/L. They do not establish labeled indication, safe initiation, absence of contraindications, medication interactions, clinical chronicity, or clinician judgment. We report missing potassium separately and distinguish albuminuria levels. Do not call these groups “Kerendia-eligible.”

In [ ]:
def group_row(label,flag,kind='observed joint-proxy segment'):
    mask=complete & d.joint.eq(1) & flag.fillna(False)
    cnt=int(mask.sum()); total=float(w[mask].sum())
    v,df0=taylor_var(mask.astype(float));tcrit=tdist.ppf(.975,df0)
    return {'kind':kind,'segment':label,'unweighted_cases':cnt,'weighted_cases':round(total),
            'count_ci_low':max(0,round(total-tcrit*np.sqrt(v))),
            'count_ci_high':round(total+tcrit*np.sqrt(v)),
            'weighted_complete_case_adult_denominator':round(den),
            'prevalence_among_complete_case_adults':total/den,
            'share_of_joint_positive':total/num if num else np.nan,
            'unstable_flag':cnt<30}
rows=[]
for g in d.g_cat.cat.categories:
    for a in d.a_cat.cat.categories:
        rows.append(group_row(f'{g} | {a}',d.g_cat.eq(g)&d.a_cat.eq(a),'GxA'))
for label,flag in [
 ('eGFR >=25',d.egfr.ge(25)),('eGFR <25',d.egfr.lt(25)),
 ('UACR <30',d.uacr.lt(30)),('UACR 30-299',d.uacr.ge(30)&d.uacr.lt(300)),('UACR >=300',d.uacr.ge(300)),
 ('Potassium <=5.0',d.potassium.le(5.0)),('Potassium >5.0',d.potassium.gt(5.0)),('Potassium missing',d.potassium.isna()),
 ('eGFR >=25 AND potassium <=5.0',d.egfr.ge(25)&d.potassium.le(5.0)),
 ('eGFR >=25 AND potassium <=5.0 AND UACR >=30',d.egfr.ge(25)&d.potassium.le(5.0)&d.uacr.ge(30)),
 ('eGFR >=25 AND potassium missing',d.egfr.ge(25)&d.potassium.isna()),
]:rows.append(group_row(label,flag))
for a in d.age_group.cat.categories:
    rows.append(group_row(f'Age {a}: eGFR >=25 AND potassium <=5.0',d.age_group.eq(a)&d.egfr.ge(25)&d.potassium.le(5.0),'age x measurements'))
for sex in ['Male','Female']:
    rows.append(group_row(f'{sex}: eGFR >=25 AND potassium <=5.0',d.sex.eq(sex)&d.egfr.ge(25)&d.potassium.le(5.0),'sex x measurements'))
segments=pd.DataFrame(rows)
segments.to_csv(ROOT/'v3_population_segments.csv',index=False)
assert segments.loc[segments.kind=='GxA','unweighted_cases'].sum()==int((d.joint==1).sum())
assert abs(segments.loc[segments.kind=='GxA','weighted_cases'].sum()-num)<20
assert segments.loc[segments.segment.isin(['Potassium <=5.0','Potassium >5.0','Potassium missing']),'unweighted_cases'].sum()==int((d.joint==1).sum())
display(segments[['kind','segment','unweighted_cases','weighted_cases','share_of_joint_positive','unstable_flag']])

## 8. Machine-readable observation catalog for future KG integration
These are **separate derived observations**, not writes to the baseline graph. Each observation includes population definition, time, data sources, methodology, units, uncertainty and caveats. Do not add these as biomedical `Drug` or `Disease` nodes.

In [ ]:
provenance={
 'dataset_cycle':'NHANES 2017-2018','population':'U.S. civilian noninstitutionalized adults age 20+',
 'joint_case_definition':'Single-measure CKD screen (eGFR <60 OR UACR >=30) AND self-reported diabetes first diagnosed at age >=30',
 'survey_design':'NHANES MEC 2-year weights; masked strata and PSUs; Taylor linearization',
 'sources':SOURCES,'baseline_kg_sha256':kg_hash,
 'limitations':['single-timepoint CKD screen; chronicity unknown','diabetes type not confirmed',
                'complete-case selection may bias estimates','observed potassium not treatment suitability',
                'not a drug-eligibility or market-sizing analysis']}
observations=[]
for _,row in segments.iterrows():
    observations.append({'observation_id':f'NHANES1718_V3_{len(observations)+1:03d}',
        'measure':'weighted_population_count','population_definition':provenance['joint_case_definition'],
        'segment':row.segment,'value':int(row.weighted_cases),'unit':'persons',
        'ci_low':int(row.count_ci_low),'ci_high':int(row.count_ci_high),
        'sample_cases':int(row.unweighted_cases),'unstable':bool(row.unstable_flag),
        'period':'2017-2018','source_ids':list(SOURCES),'baseline_sha256':kg_hash,
        'claim_status':'observed-survey-derived; not confirmed clinical eligibility'})
(ROOT/'v3_population_observations.json').write_text(json.dumps({'provenance':provenance,'observations':observations},indent=2))
print('Observations exported:',len(observations))

## 9. Automated QA and validation report
Stops on broken partitions, changed baseline, inconsistent counts or mismatch with V2. Statistical precision, diabetes classification and generalizability still require expert review.

In [ ]:
assert hashlib.sha256(kg_path.read_bytes()).hexdigest()==kg_hash
assert int((d.joint==1).sum())==314
assert checks['complete_case_unweighted']==4662
assert checks['positive_unweighted']==314
assert checks['missing_unweighted']==907
assert main['design_df']>0
assert 0<=main['prevalence_ci_low']<=main['prevalence']<=main['prevalence_ci_high']<=1
validation={'status':'PASS: internal computational checks (not external epidemiologic validation)',
 'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'baseline_sha256':kg_hash,'baseline_unchanged':True,'github_accessed':False,
 'denominator_reconciliation':checks,'primary_estimate':main,
 'gxa_partition_unweighted':int(segments.loc[segments.kind=='GxA','unweighted_cases'].sum()),
 'potassium_partition_unweighted':int(segments.loc[segments.segment.isin(['Potassium <=5.0','Potassium >5.0','Potassium missing']),'unweighted_cases'].sum()),
 'clinical_eligibility_estimated':False,
 'limitations':provenance['limitations']}
(ROOT/'v3_validation_report.json').write_text(json.dumps(validation,indent=2))
print('PASS: V3 internal QA. Outputs:',*[p.name for p in ROOT.glob('v3_*')],sep='\n - ')

## 10. Export to separate Google Drive folder (optional)
This copies only V3 outputs and the untouched baseline copy. It does not access or write to GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
target=Path('/content/drive/MyDrive/Healthcare_KG_Population_Intelligence_V3')
target.mkdir(parents=True,exist_ok=True)
for path in list(ROOT.glob('v3_*'))+[kg_path]:
    shutil.copy2(path,target/path.name)
print('Saved to',target)
print('Upload the V3 outputs ZIP here for review; we will not advance to newer NHANES cycles until validation is accepted.')

### Scientific gate before V4
- Independently reproduce estimates with a recognized survey-analysis implementation (e.g., R `survey`), and inspect missingness by demographic group.
- Check CKD-EPI laboratory standardization and whether the 2017–2018 creatinine measurements and urine albumin/creatinine support these calculations.
- Validate diabetes classification against questionnaire/medication variables; do not equate diagnosed age ≥30 with confirmed T2D.
- Consult **current** FDA prescribing information before interpreting any thresholds as indication-specific eligibility; clinical criteria and approvals can change.
- Only then consider newer NHANES cycles, and **never** pool survey weights across cycles without following NHANES guidance.